# Introduction and Biological Inspiration

The Bird Swarm Algorithm (BSA) is a nature-inspired meta-heuristic optimization algorithm that mimics the collective behavior of bird flocks.  
I uses a guided randomization mechanism to generate solutions with high diversity property.  
It was proposed by Meng et al. in 2016 and is based on three fundamental bird behaviors:

## Biological Behaviors:
1. **Foraging Behavior**: Birds search for food using their experience and social information
2. **Vigilance Behavior**: Birds remain alert for predators while competing for resources
3. **Flight Behavior**: Birds exhibit producer-scrounger dynamics during long-distance flights

# BSO Rules

1. Each bird can switch between the vigilance behavior and foraging behavior. Whether bird forages or keeps vigilance is modelled as a stochastic decision.

2. While foraging, each bird can promptly record and update its previous best experience and the swarms’ previous best experience about food patch. This experience can also be used to search for food. Social information is shared instantaneously among the whole swarm.

3. When keeping vigilance, each bird would try to move towards the centre of the swarm. This behavior can be affected by the interference induced by the competition among swarm. The birds with the higher reserves would be more likely to lie closer to the centre of the swarm than those with the lower reserves.

4. Birds would periodically fly to another site. When flying to another site, birds may often switch between producing and scrounging. The bird with the highest reserves would be a producer, while the one with the lowest reserves would be a scrounger. Other birds with reserves between the highest and lowest reserves would randomly choose to be producer and scrounger.

5. Producers actively search for food. Scroungers would randomly follow a producer to search for food.

# Algorithm Overview

BSA maintains a population of birds (candidate solutions) that iteratively improve their positions in the search space through the three behaviors mentioned above.

## Key Components:
- **Population**: N birds with D-dimensional positions
- **Personal Best**: Each bird remembers its best-found position
- **Global Best**: The best position found by the entire swarm
- **Behavioral Rules**: Mathematical models of foraging, vigilance, and flight

# Mathematical Formulations

## Population Initialization

The initial population is randomly distributed within the search bounds:

$$
X_{i,j}^0 = X_{min,j} + rand(0,1) × (X_{max,j} - X_{min,j})
$$

Where:
- $X_{i,j}^0$: Initial position of bird $i$ in dimension $j$
- $X_{min,j}, X_{max,j}$: Lower and upper bounds for dimension $j$
- $rand(0,1)$: Random number between 0 and 1

## Foraging Behavior

Birds search for food based on their personal experience and social information. The mathematical model is:

$$
X_{i,j}^{t+1} = X_{i,j}^t + C1 × rand(0,1) × (P_{i,j} - X_{i,j}^t) + C2 × rand(0,1) × (g_j - X_{i,j}^t)
$$

Where:
- $X_{i,j}^t$: Current position of bird i in dimension j at iteration t
- $P_{i,j}$: Personal best position of bird i in dimension j
- $g_j$: Global best position in dimension j
- $C1$: Cognitive acceleration coefficient (attraction to personal best)
- $C2$: Social acceleration coefficient (attraction to global best)

**Key Insight:**
- Birds with **lower fitness** values (better solutions) are **stronger** competitors and **move closer to the center** with more stable movements.

- Birds with **higher fitness** values (worse solutions) **move more randomly** and stay **farther from the center**.

**Biological Interpretation**: 
- The cognitive component represents the bird's memory of where it found good food
- The social component represents learning from the flock's collective knowledge

## Vigilance Behavior

Birds move toward the swarm center while competing with other birds. This behavior has two components:

**Swarm Center Attraction:**
$$
X_{i,j}^{t+1} = X_{i,j}^t + A1 × rand(0,1) × (Mean_j - X_{i,j}^t)
$$

**Competition Component:**
$$
X_{i,j}^{t+1} = X_{i,j}^t + A2 × randn(-1,1) × (P_{k,j} - X_{i,j}^t)
$$

**Combined Vigilance Update:**
$$
X_{i,j}^{t+1} = X_{i,j}^t + A1 × rand(0,1) × (Mean_j - X_{i,j}^t) + A2 × randn(-1,1) × (P_{k,j} - X_{i,j}^t)
$$

Where:
- $Mean_j$: Mean position of all birds in dimension j
- $k$: Randomly selected bird (k ≠ i)
- $randn(-1,1)$: Random number from uniform distribution [-1,1]

**Adaptive Parameters A1 and A2:**

$$
A_1 = \alpha_1 \cdot \exp\left(-\frac{p_{i}}{\text{sumFitness} + \epsilon} \cdot N\right)
$$

$$
A_2 = \alpha_2 \cdot \exp\left(\frac{p_{i} - p_{l}}{|p_{l} - p_{i}| + \epsilon} \cdot \frac{N \cdot p{_l}}{\text{sumFitness} + \epsilon}\right)
$$

Where:
- $p_i, p_k$: Personal best fitness values of birds $i$ and $k$
- $p_l$: Personal best fitness of randomly chosen bird $l$
- $N$: Population size
- sumFitness: Sum of all personal best fitness values
- $a1, a2$: Vigilance parameters (typically 1.0)
- $\epsilon$: Small constant to avoid division by zero (1e-10)

**Biological Interpretation**:
- $A_1$ represents the tendency to stay close to the flock center for safety
- $A_2$ represents competitive behavior between individuals
- Better-performing birds have stronger influence on others

## Flight Behavior

Flight behavior occurs periodically (every FQ iterations) and involves producer-scrounger dynamics:

**Producer Behavior (Best Bird):**
$$
X_{producer,j}^{t+1} = X_{producer,j}^t + randn(0,1) × X_{producer,j}^t
$$

**Scrounger Behavior (Worst Bird):**
$$
X_{scrounger,j}^{t+1} = X_{scrounger,j}^t + FL × rand(0,1) × (X_{producer,j}^t - X_{scrounger,j}^t)
$$

**Other Birds:**  
Other birds randomly choose between producer or scrounger behavior with 50% probability.

Where:
- $randn(0,1)$: Random number from standard normal distribution
- $FL$: Flight length parameter (typically 2.0)
- $FQ$: Flight frequency (typically every 10 iterations)

**Biological Interpretation**:
- Producers explore new areas (like lead birds in migration)
- Scroungers follow and exploit information from producers
- This mimics the information cascade effect in bird flocks

# Decision Mechanism

At each iteration, each bird makes a stochastic decision between foraging and vigilance:


> if rand(0,1) < P:  
> &nbsp;&nbsp;&nbsp;&nbsp;Apply Foraging Behavior  
> else:  
> &nbsp;&nbsp;&nbsp;&nbsp;Apply Vigilance Behavior  


Where P is the probability threshold (typically 0.8).

**Biological Interpretation**: Birds spend most time foraging (80%) but must remain vigilant for predators (20%).

# Complete Algorithm Flow

> 1. Initialize population randomly within bounds  
> 2. Evaluate fitness for all birds  
> 3. Initialize personal and global bests  
> 4. For t = 1 to MaxIterations:  
> &nbsp;&nbsp;&nbsp;&nbsp;a. For each bird i:  
> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- If rand() < P:  
> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Apply foraging behavior  
> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- Else:  
> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Apply vigilance behavior  
> &nbsp;&nbsp;&nbsp;&nbsp;b. If t mod FQ == 0: Apply flight behavior to all birds  
> &nbsp;&nbsp;&nbsp;&nbsp;c. Apply boundary constraints  
> &nbsp;&nbsp;&nbsp;&nbsp;d. Evaluate fitness for all birds  
> &nbsp;&nbsp;&nbsp;&nbsp;e. Update personal and global bests  
> 5. Return global best solution  
>   

# Problem Mapping to Cloud Load Balancing

## Biological Metaphor Translation

| Bird Behavior | Cloud Computing Equivalent | Implementation in Our Code |
|---------------|---------------------------|---------------------------|
| **Birds** | **Tasks/Cloudlets** | Population particles representing task assignments |
| **Food Sources** | **Virtual Machines (VMs)** | Destination resources for task execution |
| **Foraging** | **Task Assignment Process** | `foraging_behavior()` method |
| **Vigilance** | **Load Balancing Check** | `vigilance_behavior()` method |
| **Flight** | **Task Migration** | `flight_behavior()` method |

## Multi-Objective Function

Use a combined fitness function optimizing:
- **Minimize Makespan** (maximum completion time)
- **Maximize Resource Utilization** 
- **Minimize Degree of Imbalance (DOI)**

**Mathematical Formulation:**
$$
\text{fitness} = \alpha \cdot \text{makespan} + \beta \cdot (1 - \text{avg}_{\text{utilization}}) + \gamma \cdot \text{DOI}
$$

# Flowchart

**Raw BSO**
<div style="text-align:center">
    <img src="./assets/bso_flowchart.png" alt="flowchart for bso algorithm">
</div>

**BSO with Task Scheduling Perspective**
<div style="text-align:center">
    <img src="./assets/bso_flowchart2.png" alt="flowchart for bso algorithm">
</div>

# Comparison with Other Algorithms

| Algorithm | Inspiration | Key Difference |
|-----------|-------------|----------------|
| PSO | Bird flocking | BSA has vigilance and flight behaviors |
| GA | Evolution | BSA uses continuous position updates |
| ABC | Bee foraging | BSA includes competition and producer-scrounger dynamics |
| Grey Wolf | Wolf hunting | BSA has stochastic behavior switching |

The Bird Swarm Algorithm's strength lies in its realistic modeling of bird behavior, providing a natural balance between exploration and exploitation through its three distinct behavioral patterns.

# Implementation

## Import Libraries

In [1]:
import numpy as np
from typing import Tuple, Callable, Optional

## Bird Swarm Algorithm (BSA) implementation for optimization problems.

In [2]:
class BirdSwarmOptimizer:
    """
    Bird Swarm Algorithm (BSA) implementation for optimization problems.
    
    The BSA mimics the behavior of bird flocks with foraging, vigilance, 
    and flight behaviors to find optimal solutions.
    """
    
    def __init__(self, 
                 fitness_function: Callable[[np.ndarray], np.ndarray],
                 pop_size: int = 30,
                 dim: int = 10,
                 bounds: Tuple[float, float] = (-5.12, 5.12),
                 P: float = 0.8,
                 c1: float = 1.5,
                 c2: float = 1.5,
                 a1: float = 1.0,
                 a2: float = 1.0,
                 FL: float = 2.0,
                 FQ: int = 10):
        """
        Initialize Bird Swarm Optimizer.
        
        Args:
            fitness_function: Function to minimize f(x) -> fitness values
            pop_size: Population size (number of birds)
            dim: Problem dimension
            bounds: Search space bounds (min, max)
            P: Probability threshold for foraging vs vigilance [0,1]
            c1, c2: Cognitive and social acceleration coefficients
            a1, a2: Vigilance behavior parameters
            FL: Flight length factor for scroungers
            FQ: Flight frequency (every FQ iterations)
        """
        self.fitness_function = fitness_function
        self.pop_size = pop_size
        self.dim = dim
        self.bounds = bounds
        self.P = P
        self.c1 = c1
        self.c2 = c2
        self.a1 = a1
        self.a2 = a2
        self.FL = FL
        self.FQ = FQ
        
        # Initialize population and best solutions
        self.population = None
        self.personal_best_positions = None
        self.personal_best_fitness = None
        self.global_best_position = None
        self.global_best_fitness = None
        self.fitness_history = []
    
    def initialize_population(self) -> None:
        """Initialize random population within bounds."""
        X_min, X_max = self.bounds
        self.population = X_min + np.random.rand(self.pop_size, self.dim) * (X_max - X_min)
        
        # Initialize personal bests
        self.personal_best_positions = self.population.copy()
        self.personal_best_fitness = self.fitness_function(self.population)
        
        # Initialize global best
        best_idx = np.argmin(self.personal_best_fitness)
        self.global_best_position = self.personal_best_positions[best_idx].copy()
        self.global_best_fitness = self.personal_best_fitness[best_idx]
    
    def apply_boundary_constraints(self) -> None:
        """Ensure all birds stay within search bounds."""
        X_min, X_max = self.bounds
        self.population = np.clip(self.population, X_min, X_max)
    
    def foraging_behavior(self, bird_idx: int) -> np.ndarray:
        """
        Implement foraging behavior for a bird.
        Birds search for food based on personal and global experience.
        
        Args:
            bird_idx: Index of the bird
            
        Returns:
            New position after foraging
        """
        current_pos = self.population[bird_idx]
        personal_best = self.personal_best_positions[bird_idx]
        
        # Random coefficients
        r1 = np.random.rand(self.dim)
        r2 = np.random.rand(self.dim)
        
        # Update position: cognitive + social components
        new_pos = (current_pos + 
                  self.c1 * r1 * (personal_best - current_pos) +
                  self.c2 * r2 * (self.global_best_position - current_pos))
        
        return new_pos
    
    def vigilance_behavior(self, bird_idx: int) -> np.ndarray:
        """
        Implement vigilance behavior for a bird.
        Birds move toward swarm center while competing with others.
        
        Args:
            bird_idx: Index of the bird
            
        Returns:
            New position after vigilance behavior
        """
        current_pos = self.population[bird_idx]
        
        # Calculate swarm center
        swarm_center = np.mean(self.population, axis=0)
        
        # Select random competitor (different from current bird)
        competitors = [j for j in range(self.pop_size) if j != bird_idx]
        competitor_idx = np.random.choice(competitors)
        
        # Calculate adaptive parameters
        sum_fitness = np.sum(self.personal_best_fitness) + 1e-10
        
        A1 = self.a1 * np.exp(-self.personal_best_fitness[bird_idx] / sum_fitness * self.pop_size)
        
        fitness_diff = abs(self.personal_best_fitness[competitor_idx] - 
                          self.personal_best_fitness[bird_idx]) + 1e-10
        A2 = self.a2 * np.exp((self.personal_best_fitness[bird_idx] - 
                              self.personal_best_fitness[competitor_idx]) / fitness_diff * 
                             (self.pop_size * self.personal_best_fitness[competitor_idx] / sum_fitness))
        
        # Update position
        new_pos = (current_pos + 
                  A1 * (swarm_center - current_pos) * np.random.rand(self.dim) +
                  A2 * (self.personal_best_positions[competitor_idx] - current_pos) * 
                  np.random.uniform(-1, 1, self.dim))
        
        return new_pos
    
    def flight_behavior(self) -> None:
        """
        Implement flight behavior for the entire swarm.
        Includes producer (best bird) and scrounger (worst bird) behaviors.
        """
        current_fitness = self.fitness_function(self.population)
        
        # Identify roles
        producer_idx = np.argmin(current_fitness)
        scrounger_idx = np.argmax(current_fitness)
        
        for i in range(self.pop_size):
            if i == producer_idx:
                # Producer: explores randomly around current position
                self.population[i] += np.random.normal(0, 1, self.dim) * self.population[i]
                
            elif i == scrounger_idx:
                # Scrounger: follows the producer
                direction = self.population[producer_idx] - self.population[i]
                self.population[i] += direction * self.FL * np.random.rand(self.dim)
                
            else:
                # Other birds: randomly choose between producer or scrounger behavior
                if np.random.rand() < 0.5:
                    # Act as producer
                    self.population[i] += np.random.normal(0, 1, self.dim) * self.population[i]
                else:
                    # Act as scrounger
                    direction = self.population[producer_idx] - self.population[i]
                    self.population[i] += direction * self.FL * np.random.rand(self.dim)
    
    def update_best_solutions(self) -> None:
        """Update personal and global best solutions."""
        current_fitness = self.fitness_function(self.population)
        
        # Update personal bests
        improved_mask = current_fitness < self.personal_best_fitness
        self.personal_best_positions[improved_mask] = self.population[improved_mask].copy()
        self.personal_best_fitness[improved_mask] = current_fitness[improved_mask]
        
        # Update global best
        best_idx = np.argmin(self.personal_best_fitness)
        if self.personal_best_fitness[best_idx] < self.global_best_fitness:
            self.global_best_position = self.personal_best_positions[best_idx].copy()
            self.global_best_fitness = self.personal_best_fitness[best_idx]
    
    def optimize(self, max_iter: int = 100, verbose: bool = False) -> Tuple[np.ndarray, float]:
        """
        Run the Bird Swarm Algorithm optimization.
        
        Args:
            max_iter: Maximum number of iterations
            verbose: Whether to print progress
            
        Returns:
            Tuple of (best_position, best_fitness)
        """
        # Initialize population
        self.initialize_population()
        self.fitness_history = [self.global_best_fitness]
        
        for iteration in range(max_iter):
            # Apply behaviors to each bird
            for i in range(self.pop_size):
                if np.random.rand() < self.P:
                    # Foraging behavior
                    self.population[i] = self.foraging_behavior(i)
                else:
                    # Vigilance behavior
                    self.population[i] = self.vigilance_behavior(i)
            
            # Apply flight behavior periodically
            if iteration % self.FQ == 0 and iteration > 0:
                self.flight_behavior()
            
            # Apply boundary constraints
            self.apply_boundary_constraints()
            
            # Update best solutions
            self.update_best_solutions()
            
            # Store fitness history
            self.fitness_history.append(self.global_best_fitness)
            
            # Print progress
            if verbose and iteration % 10 == 0:
                print(f"Iteration {iteration}: Best Fitness = {self.global_best_fitness:.6f}")
        
        return self.global_best_position, self.global_best_fitness

## Common test functions

In [ ]:
# 
def sphere_function(x: np.ndarray) -> np.ndarray:
    """Sphere function: f(x) = sum(x_i^2). Global minimum: f(0) = 0"""
    return np.sum(x**2, axis=1)


def rosenbrock_function(x: np.ndarray) -> np.ndarray:
    """Rosenbrock function: f(x) = sum(100*(x_{i+1} - x_i^2)^2 + (1 - x_i)^2)"""
    if x.ndim == 1:
        x = x.reshape(1, -1)
    
    result = np.zeros(x.shape[0])
    for i in range(len(result)):
        xi = x[i]
        result[i] = np.sum(100.0 * (xi[1:] - xi[:-1]**2)**2 + (1 - xi[:-1])**2)
    
    return result


def rastrigin_function(x: np.ndarray) -> np.ndarray:
    """Rastrigin function: f(x) = A*n + sum(x_i^2 - A*cos(2*pi*x_i))"""
    A = 10
    n = x.shape[1] if x.ndim > 1 else len(x)
    if x.ndim == 1:
        x = x.reshape(1, -1)
    
    return A * n + np.sum(x**2 - A * np.cos(2 * np.pi * x), axis=1)

## Optimization Runner

In [ ]:
def run_optimization_example():
    """Example usage of the Bird Swarm Algorithm."""
    
    print("=== Bird Swarm Algorithm Optimization Examples ===\n")
    
    # Test functions and their properties
    test_functions = [
        ("Sphere Function", sphere_function, (-5.12, 5.12), 30),
        ("Rosenbrock Function", rosenbrock_function, (-5, 10), 10),
        ("Rastrigin Function", rastrigin_function, (-5.12, 5.12), 30)
    ]
    
    for func_name, func, bounds, dim in test_functions:
        print(f"--- Optimizing {func_name} ---")
        
        # Create optimizer
        optimizer = BirdSwarmOptimizer(
            fitness_function=func,
            pop_size=50,
            dim=dim,
            bounds=bounds,
            P=0.8,
            c1=1.5,
            c2=1.5,
            a1=1.0,
            a2=1.0,
            FL=2.0,
            FQ=10
        )
        
        # Run optimization
        best_pos, best_fitness = optimizer.optimize(max_iter=200, verbose=False)
        
        print(f"Best Fitness: {best_fitness:.6f}")
        print(f"Best Position: {best_pos[:5]}...")  # Show first 5 dimensions
        print(f"Iterations: {len(optimizer.fitness_history)}")
        print()


if __name__ == "__main__":
    run_optimization_example()

=== Bird Swarm Algorithm Optimization Examples ===

--- Optimizing Sphere Function ---
Best Fitness: 0.000000
Best Position: [ 1.05452481e-06 -7.02246028e-07 -2.30488869e-07 -1.06069716e-06
  1.94962776e-06]...
Iterations: 201

--- Optimizing Rosenbrock Function ---
Best Fitness: 6.272067
Best Position: [0.80381944 0.63050552 0.3980471  0.15233137 0.00522658]...
Iterations: 201

--- Optimizing Rastrigin Function ---
Best Fitness: 71.881087
Best Position: [-9.95603783e-01 -1.00171718e+00 -9.78704428e-01  1.01298177e+00
  5.98805825e-04]...
Iterations: 201



# References

A new bio-inspired optimization algorithm: Bird Swarm Algorithm

A binary Bird Swarm Optimization based load balancing algorithm for cloud computing environment